Parameters

In [1]:
# Parameters
parameters_dict = {
    'model_validation_strategy': 'k_fold_cross_validation',  # Options: 'k_fold_cross_validation', 'train_val_split'
    'number_of_cases_in_sample': 1000,
    'number_of_epochs_for_training': 20,
    'batch_size_for_training': 16,
    'top_n_performing_labels_to_display': 10,
    'numbers_of_cases_to_analyze_for_interpretability': 200,
    'minimum_confidence_for_wrong_prediction_inclusion': 0.5,
    'number_of_highest_probability_predictions_per_step': 5,
    'maximum_number_of_confident_mistakes_to_return': 25,
    'maximum_validation_sequences_to_process_for_analysis_of_incorrect': 100,
    'num_best_and_worst_cases_to_display': 5,
    'k_fold_num_folds': 5,        # Number of folds for cross-validation
    'k_fold_shuffle': True,       # Shuffle before splitting
    'save_cv_plots': True,        # Whether to save plots of CV metrics
    'cv_plot_filename': 'cross_validation_results.png',
    'save_each_fold_model': True,  # Save every fold’s model state if True
    'save_best_fold_only': False,  # Save only the best fold’s weights if True
    'new_dataset_creation': True,
    'sequence_data_mode': 'export',  # Options: 'current', 'export', 'import'    
    'random_seed': 42,
    'weighted_loss_function': True,
    'sampling_strategy': 'oversample_long_sequences', #options are 'uniform' and 'oversample_long_sequences' and 'Balanced' and None
    'skip_duplicate_labels': False,
    'regex_labels_mode': 'start_from_scratch', #options are 'load_in' or 'start_from_scratch'
    'folder_name_directory': 'full_data'
}

# Imports

In [2]:
from collections import Counter
from collections import defaultdict
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from openpyxl.utils.dataframe import dataframe_to_rows
from operator import sub
from sklearn.preprocessing import LabelEncoder
from pathlib import Path
import types
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import contextlib
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
from transformers import AutoModel, AutoTokenizer
import gc
import h5py  # For efficient storage of large arrays
import hashlib
import json
import matplotlib.pyplot as plt
import numpy as np
import time
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
import openpyxl
import os
import pandas as pd
import pickle
import random
import re
import seaborn as sns
import string
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import shutil
import xlsxwriter
from scipy.sparse import csr_matrix
from captum.attr import IntegratedGradients, LayerConductance, LayerActivation
from captum.attr import visualization as viz

In [3]:
# Set device globally
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

Using device: cuda
GPU: NVIDIA RTX A5000
CUDA Version: 12.4


In [4]:
def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [5]:
set_global_seed(parameters_dict['random_seed'])

In [ ]:
from config import get_model_config, COMPLETION_LABELS
from data.load_data import mount_drive, load_data
from data.process_data import append_prefix_to_proceeding_ids, filter_proceedings, remove_multi_defendant_cases, create_combined_dockets, filter_to_magistrate_started_cases, remove_extradition_cases
from data.sequence_builder import create_enhanced_training_sequences_cached, get_proceeding_sequence_for_defendant
from regex_labels.label_generator_and_cache_manager import generate_and_cache_regex_labels
from models.transformer_creation import create_model_from_data
from training.trainer import train_case_transformer_with_mlb
from training.label_processor import EnhancedLabelProcessor
from training.evaluation import evaluate_with_teacher_forcing
from analysis.cooccurrence import export_cooccurrence_from_valid_completed_cases, create_cooccurrence_matrix_excel
from analysis.frequency_analysis import analyze_raw_label_frequencies_combined, analyze_raw_label_frequencies_singular, print_label_distribution_by_court
from analysis.diagnostic import comprehensive_training_diagnostic
from models.baseline_model import run_baseline, convert_baseline_results_to_table_format
from utils.cache_directory_utils import CacheDirectoryManager
from utils.model_memory_utils import analyze_model_memory, debug_oom_sample
from models.embeddings import extract_and_cache_bert_embeddings_streaming
from analysis.byclerk import analyze_clerk_data
from models.interpretability import CaseTransformerInterpretability, CaptumWrapper
from models.most_confident_wrong_answer_analyzer import run_confident_misclassification_analysis
from data.k_fold_methods import KFoldCVRunner
from data.dataset_creation import sample_from_the_longest_cases, export_long_cases_to_xlsx, sample_uniformly, export_uniform_cases_to_xlsx
from analysis.tablegenerator import TableGenerator
from data.balanced_dataset_creation import create_balanced_dataset, export_balanced_cases_to_xlsx
from models.export_model import export_model_complete

Config initialized with device: cuda


In [7]:
%load_ext autoreload
%autoreload 2
from notebook_helpers import (
    # Data filtering helpers
    filter_to_completed_cases,
    build_case_counts_df,
    filter_zero_label_proceedings,
    generate_cooccurrence_matrix_excel,
    filter_proceedings_to_end_of_case,
    
    # Data transformation helpers
    transform_labels_to_binary,
    convert_data_to_format,
    
    # Sequence processing helpers
    filter_sequences_with_meaningful_labels,
    split_train_validation,
    prepare_sequences_and_labels,
    
    # Case info extraction
    extract_case_info_from_sequence,
    
    # Prediction helpers
    build_prediction_prefix,
    predict_case_labels,
    predict_step,
    
    # Metrics calculation helpers
    calculate_metrics,
    calculate_case_metrics,
    collect_labels_from_binary,
    count_processed_steps,
    
    # Analysis helpers
    perform_frequency_analysis,
    debug_model_sample,
    
    # Training helpers
    train_model,
    
    # Evaluation helpers
    prepare_validation_data_for_eval,
    evaluate_model,
    print_micro_averaged_metrics,
    print_top_performing_labels,
    
    # Case-level evaluation helpers
    evaluate_case_level,
    print_case_level_metrics,
    save_case_level_results,
    print_best_worst_cases,
    print_f1_distribution,
    calculate_label_metrics_from_cases,
    save_label_metrics_to_excel,
    
    # Advanced analysis helpers
    run_diagnostic_analysis,
    run_baseline_comparison,
    run_confident_misclassification,
    
    # Interpretability helpers
    initialize_interpretability_analyzer,
    select_samples_for_analysis,
    get_model_prediction,
    compute_attributions,
    visualize_attributions,
    get_and_save_feature_summary,
    analyze_single_sample,
    run_attribution_analysis,
    aggregate_component_importance,
    save_and_visualize_aggregate_importance,
    find_sample_with_label,
    analyze_label_specific_features,
    create_label_feature_comparison,
    run_interpretability_analysis,
    analyze_confident_misclass_for_label,
    compute_global_attributions_table,
    
    # Sample prediction helpers
    print_step_prediction_analysis,
    analyze_sample_predictions,

    # ablation study helpers
    create_standard_ablation_configurations,
    run_ablation_study,
    compare_ablation_table
)

In [8]:
folder = rf'/data/krosen5/cache/transformers/{parameters_dict["folder_name_directory"]}/'
if not os.path.exists(folder):
    os.makedirs(folder)
model_cache_manager = CacheDirectoryManager(base_path=folder)

In [9]:
table_generator = TableGenerator(
    output_dir=model_cache_manager.get_cache_dir('models')
)

# Main Method

## 1. Mount and load data

In [10]:
drive_root_folder = mount_drive('tulane-courts-shared/v2')
court_data = load_data(drive_root_folder)

✓ Found Google Drive mount at /data/krosen5/gdrive-all
✓ Google Drive mount is accessible
✓ Data directory found: /data/krosen5/gdrive-all/tulane-courts-shared/v2


In [11]:
model_configuration = get_model_config()

In [ ]:
court_data = append_prefix_to_proceeding_ids(court_data)
court_data = filter_proceedings(court_data)
crawled_original_data_folder_path = Path("/data/krosen5/gdrive-all/tulane-courts-shared/crawled")
court_data, multi_defendant_cases  = remove_multi_defendant_cases(court_data, crawled_original_data_folder_path)

Removed 2095 CDC and 1919 mag cases (including corresponding matches)


In [13]:
print("   Creating combined dockets...")
court_data['combined_dockets'] = create_combined_dockets(court_data)

   Creating combined dockets...


In [14]:
print("Removing cases that start in CDC and have no MAG proceedings...")
court_data = filter_to_magistrate_started_cases(court_data)

Removing cases that start in CDC and have no MAG proceedings...
Starting with 128475 total cases in combined_dockets
Found 123589 magistrate-started cases
  Filtered mag proceedings: 154945 -> 154741
  Filtered mag_dockets: 123748 -> 123589
  Filtered mag_charges: 284218 -> 283964
  Filtered cdc proceedings: 168073 -> 160273
  Filtered cdc_dockets: 25578 -> 20773
  Filtered cdc_charges: 61584 -> 50334
  Filtered defendants: 75526 -> 71099
Filtering complete. Dataset now contains only magistrate-started cases.


In [15]:
print("Removing cases that involve some sort of extradition")
court_data = remove_extradition_cases(court_data)
court_data['combined_dockets'] = create_combined_dockets(court_data)


Removing cases that involve some sort of extradition
Starting extradition case removal...
  Found 2049 magistrate cases with extradition indicators
  Found 657 CDC cases with extradition indicators
  Filtered mag proceedings: 154741 -> 144096
  Filtered cdc proceedings: 160273 -> 149674
  Filtered mag_dockets: 123589 -> 120998
  Filtered cdc_dockets: 20773 -> 20015
  Filtered mag_charges: 283964 -> 280050
  Filtered cdc_charges: 50334 -> 48631
  Filtered defendants: 71099 -> 69296
  Removed combined_dockets (will need to be recreated)
Extradition case removal complete.
  Removed 2591 magistrate cases
  Removed 767 CDC cases


In [16]:
def clean_the_dictionary(dictionary):
    elements_to_remove = model_configuration.EXCLUDED_LABELS
    cleaned_dict = {}
    
    for key, value in dictionary.items():
        if isinstance(value, list):
            # Remove unwanted elements while preserving order
            cleaned_list = [item for item in value if item not in elements_to_remove]
            # Only keep the key if the list is not empty
            if cleaned_list:
                cleaned_dict[key] = cleaned_list
        else:
            # Keep non-list values as they are
            cleaned_dict[key] = value
    
    return cleaned_dict

In [17]:
print("2. Processing regex labels...")
labels_cache = model_cache_manager.get_cache_dir('labels')
cached_outcome_labels = generate_and_cache_regex_labels(court_data, labels_cache, parameters_dict['regex_labels_mode'], cache_manager=model_cache_manager)
cached_outcome_labels_v2 = clean_the_dictionary(cached_outcome_labels)    
print(f"   Original labels count: {len(cached_outcome_labels)}")
print(f"   Cleaned labels count: {len(cached_outcome_labels_v2)}")

2. Processing regex labels...


  0%|          | 0/144096 [00:00<?, ?it/s]

100%|██████████| 149674/149674 [24:12<00:00, 103.02it/s]


   Original labels count: 293770
   Cleaned labels count: 164796


In [18]:
print("2b. Filtering out proceedings with zero labels...")
court_data, cached_outcome_labels_v2 = filter_zero_label_proceedings(
    court_data, 
    cached_outcome_labels_v2
)
print(f"   Final labels count after filtering: {len(cached_outcome_labels_v2)}")
court_data['combined_dockets'] = create_combined_dockets(court_data)

2b. Filtering out proceedings with zero labels...
Total proceedings in cache: 164796
Proceedings with labels: 164796
Proceedings with 0 labels: 0
Removed 59801 MAG proceedings with 0 labels
Removed 69173 CDC proceedings with 0 labels
   Final labels count after filtering: 164796


In [22]:
print("3. Setting up model...")
computation_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"   Using device: {computation_device}")    
case_outcome_model = create_model_from_data(court_data, model_cache_manager.get_cache_dir('bert'), cache_manager=model_cache_manager)
analyze_model_memory(case_outcome_model)

3. Setting up model...
   Using device: cuda
Analyzing date range in data...
Mag data - Date range: 2013-12-07 00:00:00 to 2025-12-04 00:00:00, Count: 84291
CDC data - Date range: 2018-03-07 00:00:00 to 2025-12-30 00:00:00, Count: 80501
Overall date range in data: 2013 - 2025
Date range updated: 2013 to 2025
Date range in data: 2013 to 2025
Using MIN_YEAR=2013, NUM_YEARS=13
Updated temporal dimensions: 156 features (12 months × 13 years)
Creating charge vocabulary from charges DataFrame with 2139 unique charges
Found 5 unique magistrate sections in mag data.
Creating magistrate vocabulary of size 5
Checking BERT embeddings cache...
BERT embeddings cache found.

MODEL MEMORY ANALYSIS

Total parameters:           12,761,263
  Trainable params:         12,761,263
  Non-trainable params:     0
Parameter memory footprint: 48.68 MB
Buffer memory footprint:    0.00 MB
Total model memory:         48.68 MB

Layer-wise parameter counts:
  text_emb                                  0 params (0.00 

In [23]:
print_label_distribution_by_court(
    court_data,
    cached_outcome_labels_v2,
    model_configuration.LABEL_NAMES,
    mag_key="mag",
    cdc_key="cdc",
    proceeding_col="proceeding_id")

LABEL DISTRIBUTION BY COURT TYPE
Label                                              MAG      CDC      Total    MAG %    CDC %   
--------------------------------------------------------------------------------
FIRST_APPEARANCE_HEARING_OCCURRED                  40,909   0        40,909   100.0    0.0     
BOND_SET                                           39,596   8,634    48,230   82.1     17.9    
BOND_NOT_SET                                       748      6,137    6,885    10.9     89.1    
DEFENDANT_RELEASED_ON_OWN_RECOGNIZANCE             9,645    2,590    12,235   78.8     21.2    
THE_COURT_APPOINTED_THE_ORLEANS_PUBLIC_DEFENDER    30,006   6,969    36,975   81.2     18.8    
ANKLE_MONITORING_IMPOSED                           629      236      865      72.7     27.3    
DOMESTIC_STAY_AWAY_ORDER_IMPOSED                   6,606    2        6,608    100.0    0.0     
CHARGES_ACCEPTED                                   2,180    3        2,183    99.9     0.1     
CHARGES_REFUSED       

defaultdict(<function analysis.frequency_analysis.print_label_distribution_by_court.<locals>.<lambda>()>,
            {'FIRST_APPEARANCE_HEARING_OCCURRED': {'mag': 40909, 'cdc': 0},
             'BOND_SET': {'mag': 39596, 'cdc': 8634},
             'THE_COURT_APPOINTED_THE_ORLEANS_PUBLIC_DEFENDER': {'mag': 30006,
              'cdc': 6969},
             'CONTINUANCE_OCCURRED': {'mag': 13240, 'cdc': 30528},
             'CHARGES_REFUSED': {'mag': 22238, 'cdc': 3851},
             'PROBABLE_CAUSE_NOT_FOUND': {'mag': 1755, 'cdc': 118},
             'PROBABLE_CAUSE_FOUND': {'mag': 16414, 'cdc': 2387},
             'DEFENDANT_RELEASED_ON_OWN_RECOGNIZANCE': {'mag': 9645,
              'cdc': 2590},
             'DEFENDANT_IN_CUSTODY_AND_NOT_IN_COURT': {'mag': 643,
              'cdc': 9058},
             'DOMESTIC_STAY_AWAY_ORDER_IMPOSED': {'mag': 6606, 'cdc': 2},
             'CHARGES_ACCEPTED': {'mag': 2180, 'cdc': 3},
             'BOND_NOT_SET': {'mag': 748, 'cdc': 6137},
             'P

## 2. Instantiate model

In [24]:
computation_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_cache_directory = model_cache_manager.get_cache_dir('bert')
proceeding_index_file = bert_cache_directory / "proceeding_index.pkl"
# only build the index (and embeddings.h5) if it doesn't exist
print(f"Checking for cached BERT embeddings in {bert_cache_directory}")
if not proceeding_index_file.exists():
   extract_and_cache_bert_embeddings_streaming(
        court_data,
        cache_dir=bert_cache_directory,
        model_name="bert-base-uncased",
        out_dim=768,
        batch_size=32,
        device=computation_device,
        force_recompute=False,
        cache_manager=model_cache_manager
    )

Checking for cached BERT embeddings in /data/krosen5/cache/transformers/full_data/bert_streaming


In [25]:
print("Creating model from data")
case_outcome_model = create_model_from_data(court_data, bert_cache_directory, cache_manager=model_cache_manager)

Creating model from data
Analyzing date range in data...
Mag data - Date range: 2013-12-07 00:00:00 to 2025-12-04 00:00:00, Count: 84291
CDC data - Date range: 2018-03-07 00:00:00 to 2025-12-30 00:00:00, Count: 80501
Overall date range in data: 2013 - 2025
Date range updated: 2013 to 2025
Date range in data: 2013 to 2025
Using MIN_YEAR=2013, NUM_YEARS=13
Updated temporal dimensions: 156 features (12 months × 13 years)
Creating charge vocabulary from charges DataFrame with 2139 unique charges
Found 5 unique magistrate sections in mag data.
Creating magistrate vocabulary of size 5
Checking BERT embeddings cache...
BERT embeddings cache found.


## 3. Build list of (mag_num, case_id, defendant_id) tuples

In [26]:
print("Building list of defendant cases for training")
combined_dockets_df = court_data['combined_dockets']
defendant_case_identifiers = []
for _, row in combined_dockets_df.iterrows():
    case_identifier = (row.get('mag_num'), row.get('case_id'), row.get('defendant_id'))
    defendant_case_identifiers.append(case_identifier)
valid_defendant_case_ids = [
    case_id for case_id in defendant_case_identifiers 
    if pd.notna(case_id[2]) and (pd.notna(case_id[0]) or pd.notna(case_id[1]))
]

Building list of defendant cases for training


## 4. Sample a subset and create sequences with prior-case features


In [27]:
print("Filtering to completed cases only for training")
completed_defendant_cases = filter_to_completed_cases(
    court_data, 
    valid_defendant_case_ids, 
    cached_outcome_labels_v2
)
print(f"Found {len(completed_defendant_cases)} completed cases for training")

Filtering to completed cases only for training


Filtering to completed cases: 100%|██████████| 120998/120998 [10:04<00:00, 200.25it/s]

Found 23980 completed cases for training


In [28]:
# Pre-filter valid cases first
print("Pre-filtering valid cases...")
_counts = build_case_counts_df(court_data, cached_outcome_labels_v2)

_df_cases = pd.DataFrame(completed_defendant_cases,
                         columns=['mag_num','case_id','defendant_id'])

_tagged = (_df_cases.merge(_counts, on=['mag_num','case_id','defendant_id'], how='left')
                    .assign(valid=lambda d: d['valid'].fillna(False)))

valid_completed_defendant_cases = [tuple(x) for x in
    _tagged.loc[_tagged['valid'], ['mag_num','case_id','defendant_id']].to_numpy()]

Pre-filtering valid cases...


In [29]:
training_case_sample = valid_completed_defendant_cases

In [38]:
#get me the first proceeding sequence for the first defendant in the list
first_case = training_case_sample[0]
first_case_sequence, first_case_labels = create_enhanced_training_sequences_cached(case_sample=[first_case], data=court_data, cached_labels=cached_outcome_labels_v2)


Processing cases:   0%|          | 0/1 [00:00<?, ?it/s]

Processing cases: 100%|██████████| 1/1 [00:00<00:00,  2.19it/s]


In [60]:
def export_proceeding_level_data_from_sequences(case_sample, court_data, cached_labels, 
                                                 output_path, config_name="",
                                                 skip_duplicate_labels=False, 
                                                 apply_truncation=False):
    """
    Export detailed proceeding-level data from sequences.
    
    Args:
        case_sample: List of (mag_num, case_id, defendant_id) tuples
        court_data: Court data dictionary
        cached_labels: Dictionary of proceeding_id -> labels
        output_path: Path to save the CSV file
        config_name: Name of configuration for display
        skip_duplicate_labels: Whether to skip duplicate labels
        apply_truncation: Whether to apply truncation at case conclusion
    """
    print(f"\n{'='*80}")
    print(f"Exporting: {config_name}")
    print(f"Skip Duplicates: {skip_duplicate_labels}, Apply Truncation: {apply_truncation}")
    print(f"{'='*80}")
    
    all_output_data = []
    global_seq_idx = 0
    
    # Create sequences for all cases at once
    sequences, label_sequences_raw = create_enhanced_training_sequences_cached(
        case_sample=case_sample,
        data=court_data, 
        cached_labels=cached_labels,
        skip_duplicate_labels=skip_duplicate_labels
    )
    
    # Apply truncation if requested
    if apply_truncation:
        sequences, label_sequences_raw = filter_proceedings_to_end_of_case(
            sequences,
            label_sequences_raw
        )
    
    # Process each sequence
    for i, (seq, label_seq) in enumerate(tqdm(zip(sequences, label_sequences_raw), 
                                                total=len(sequences),
                                                desc=f"Processing {config_name}")):
        mag_num, case_id, defendant_id = case_sample[i]
        
        for step_idx, (proc_dict, step_labels) in enumerate(zip(seq, label_seq)):
            # Get charge information
            charges_dict = proc_dict.get('charges', {})
            charge_names = [charge_info.get('name', '') for charge_info in charges_dict.values()]
            charge_names_str = ' | '.join(charge_names) if charge_names else ''
            
            # Get date information
            date_val = proc_dict.get('date')
            if pd.notna(date_val):
                if isinstance(date_val, str):
                    date_val = pd.to_datetime(date_val, errors='coerce')
            
            all_output_data.append({
                'sequence_idx': global_seq_idx,
                'step_idx': step_idx,
                'mag_num': mag_num if pd.notna(mag_num) else '',
                'case_id': case_id if pd.notna(case_id) else '',
                'defendant_id': defendant_id if pd.notna(defendant_id) else '',
                'proceeding_id': proc_dict.get('proceeding_id', ''),
                'date': date_val,
                'date_month_year': proc_dict.get('date_month_year', ''),
                'proceeding_type': proc_dict.get('type', ''),
                'text': str(proc_dict.get('text', ''))[:1000] if proc_dict.get('text') else '',
                'docket_id': proc_dict.get('docket_id', ''),
                'original_bond': proc_dict.get('original_bond', ''),
                'sum_of_bonds': proc_dict.get('sum_of_bonds', ''),
                'race_id': proc_dict.get('race_id', ''),
                'gender_id': proc_dict.get('gender_id', ''),
                'age': proc_dict.get('age', ''),
                'magistrate_section': proc_dict.get('magistrate_section', ''),
                'charges': charge_names_str,
                'num_charges': len(charges_dict),
                'labels': '|'.join(step_labels) if step_labels else 'NO_LABELS',
                'label_count': len(step_labels),
                'prior_case_count': proc_dict.get('prior_case_count', 0),
            })
        
        global_seq_idx += 1
    
    print(f"Creating DataFrame with {len(all_output_data)} proceedings...")
    output_df = pd.DataFrame(all_output_data)
    
    # Sort by sequence and step
    output_df = output_df.sort_values(['sequence_idx', 'step_idx']).reset_index(drop=True)
    
    # Save to CSV
    output_df.to_csv(output_path, index=False)
    print(f"✅ Saved to: {output_path}")
    print(f"   Total proceedings: {len(output_df)}")
    print(f"   Total sequences: {output_df['sequence_idx'].nunique()}")
    print(f"   Avg proceedings per sequence: {len(output_df) / output_df['sequence_idx'].nunique():.2f}")
    print(f"   Total labels: {output_df['label_count'].sum()}")
    print(f"   Avg labels per proceeding: {output_df['label_count'].mean():.2f}")
    print(f"   Proceedings with labels: {(output_df['label_count'] > 0).sum()}")
    print(f"   Proceedings without labels: {(output_df['label_count'] == 0).sum()}")
    
    return output_df


In [ ]:
# Setup output directory
output_dir = model_cache_manager.get_cache_dir('datasets')
output_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Dictionary to store dataframes for comparison
all_dataframes = {}

# ====================================================================================================
# CONFIGURATION 0: PRE-FILTERING (RAW DATA FROM SELECTED CASES)
# ====================================================================================================
print("CONFIGURATION 0: PRE-FILTERING (RAW DATA FROM SELECTED CASES)")
if not parameters_dict['sequence_data_mode'] == 'import':
    config0_path = output_dir / f'config0_pre_filtering_raw_{timestamp}.csv'
    
    print("\nExporting pre-filtering data...")
    print(f"Processing {len(training_case_sample)} cases...")
    
    output_data_raw = []
    
    for mag_num, case_id, def_id in tqdm(training_case_sample, desc="Processing cases (pre-filtering)"):
        # Use the raw function that doesn't apply ML filtering
        case_sequences = get_proceeding_sequence_for_defendant(
            def_id, mag_num, case_id, court_data
        )
        
        # Extract ALL proceedings without filtering
        for case in case_sequences:
            for proc in case['proceedings']:
                proc_id = proc.get('proceeding_id')
                labels = cached_outcome_labels_v2.get(proc_id, [])
                
                # Get charge information
                charges_dict = proc.get('charges', {})
                charge_names = [charge_info.get('name', '') for charge_info in charges_dict.values()]
                charge_names_str = ' | '.join(charge_names) if charge_names else ''
                
                output_data_raw.append({
                    'mag_num': mag_num if pd.notna(mag_num) else '',
                    'case_id': case_id if pd.notna(case_id) else '',
                    'defendant_id': def_id,
                    'proceeding_id': proc_id,
                    'date': proc.get('date'),
                    'date_month_year': proc.get('date_month_year', ''),
                    'proceeding_type': proc.get('type'),
                    'text': proc.get('text', '')[:1000] if proc.get('text') else '',
                    'docket_id': proc.get('docket_id', ''),
                    'original_bond': proc.get('original_bond', ''),
                    'sum_of_bonds': proc.get('sum_of_bonds', ''),
                    'race_id': proc.get('race_id', ''),
                    'gender_id': proc.get('gender_id', ''),
                    'age': proc.get('age', ''),
                    'magistrate_section': proc.get('magistrate_section', ''),
                    'charges': charge_names_str,
                    'num_charges': len(charges_dict),
                    'labels': '|'.join(labels) if labels else 'NO_LABELS',
                    'label_count': len(labels)
                })
    
    # Create DataFrame
    print("\nCreating DataFrame...")
    df_config0 = pd.DataFrame(output_data_raw)
    
    # Sort by defendant and date
    df_config0 = df_config0.sort_values(['defendant_id', 'date']).reset_index(drop=True)
    
    # Save to CSV
    df_config0.to_csv(config0_path, index=False)
    all_dataframes['config0_pre_filtering'] = df_config0
    
    print("\n" + "="*80)
    print("PRE-FILTERING EXPORT COMPLETE")
    print("="*80)
    print(f"✓ File saved to: {config0_path}")
    print(f"\nStatistics:")
    print(f"  - Total cases: {len(training_case_sample)}")
    print(f"  - Total proceedings: {len(df_config0)}")
    print(f"  - Proceedings with labels: {len(df_config0[df_config0['label_count'] > 0])}")
    print(f"  - Average labels per proceeding: {df_config0['label_count'].mean():.2f}")
    print(f"  - Unique defendants: {df_config0['defendant_id'].nunique()}")
    
    # Print label distribution
    print(f"\nTop 10 Most Common Labels:")
    all_labels = []
    for labels_str in df_config0[df_config0['labels'] != 'NO_LABELS']['labels']:
        all_labels.extend(labels_str.split('|'))
    
    label_counts = Counter(all_labels)
    for label, count in label_counts.most_common(10):
        print(f"  {label}: {count}")
    
    print("="*80)

if not parameters_dict['sequence_data_mode'] == 'import':
    config1_path = output_dir / f'config1_all_data_{timestamp}.csv'
    df_config1 = export_proceeding_level_data_from_sequences(
        case_sample=training_case_sample,
        court_data=court_data,
        cached_labels=cached_outcome_labels_v2,
        output_path=config1_path,
        config_name="Configuration 1: All Data",
        skip_duplicate_labels=False,
        apply_truncation=False
    )
    all_dataframes['config1_all_data'] = df_config1

# ============================================================================
# CONFIGURATION 2: Truncated + Skip Duplicate Labels
# ============================================================================
print("\n" + "="*80)
print("CONFIGURATION 2: Truncated + Skip Duplicate Labels")
print("="*80)

if not parameters_dict['sequence_data_mode'] == 'import':
    config2_path = output_dir / f'config2_truncated_skip_dups_{timestamp}.csv'
    df_config2 = export_proceeding_level_data_from_sequences(
        case_sample=training_case_sample,
        court_data=court_data,
        cached_labels=cached_outcome_labels_v2,
        output_path=config2_path,
        config_name="Configuration 2: Truncated + Skip Duplicates",
        skip_duplicate_labels=True,
        apply_truncation=True
    )
    all_dataframes['config2_truncated_skip_dups'] = df_config2

# ============================================================================
# CONFIGURATION 3: Truncated without Skip Duplicate Labels
# ============================================================================
print("\n" + "="*80)
print("CONFIGURATION 3: Truncated without Skip Duplicate Labels")
print("="*80)

if not parameters_dict['sequence_data_mode'] == 'import':
    config3_path = output_dir / f'config3_truncated_no_skip_{timestamp}.csv'
    df_config3 = export_proceeding_level_data_from_sequences(
        case_sample=training_case_sample,
        court_data=court_data,
        cached_labels=cached_outcome_labels_v2,
        output_path=config3_path,
        config_name="Configuration 3: Truncated without Skip Duplicates",
        skip_duplicate_labels=False,
        apply_truncation=True
    )
    all_dataframes['config3_truncated_no_skip'] = df_config3


Exporting: Configuration 1: All Data
Skip Duplicates: False, Apply Truncation: False


Processing cases:   0%|          | 0/23665 [00:00<?, ?it/s]

Processing cases:   3%|▎         | 744/23665 [06:11<3:10:44,  2.00it/s]


KeyboardInterrupt: 